# Example 11.4 / 1: Domain-Safe Formulation

Companion notebook for **11.4 Pitfalls of Nonlinear Programming** in the AMPL lab Documentation, section "Domain-Safe Formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Domain-Safe Formulation

For a term $x/(1-x/L)$, the denominator vanishes at $x=L$. A valid model needs a physical limit below that singularity, not merely a convenient initial value.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param limit > 0;             # capacity; cost is singular at Flow = limit
param demand >= 0, < limit;  # flow to serve, below the limit

var Flow >= 0, <= 0.99 * limit, := 0.5 * limit;  # upper bound keeps the denominator positive

minimize CongestionCost:  # grows without bound near the limit
    Flow / (1 - Flow / limit);

subject to ServeDemand:  # serve at least the demand
    Flow >= demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["limit"] = 10
ampl.param["demand"] = 6

In [5]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(round(ampl.var["Flow"].value(), 6))
print(round(ampl.con["ServeDemand"].slack(), 6))

solved
6.0
0.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param limit := 10;
param demand := 6;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
solve > ipopt.log;
display Flow, ServeDemand.slack;

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
Flow = 6
ServeDemand.slack = 0



### Pure AMPL: data.dat and model.run

The singularity is concrete: at `Flow = limit = 10` the denominator is zero and AMPL cannot
evaluate the objective. That is why the bound stops at `0.99 * limit`:

In [9]:
%%writefile data.dat
param limit := 10;
param demand := 6;

Overwriting data.dat


In [10]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [11]:
%%ampl_eval
# model.run
let Flow := 10;

In [12]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display CongestionCost;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
		can't compute 10/0


### Another way in amplpy

A `solved` status does not make a decision usable. If the demand constraint is omitted, here by
dropping it with the amplpy `drop()` method, Ipopt still reports `solved`, and the optimal flow is
`0`: correct mathematics for the wrong model.

In [13]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [14]:
# data.py
ampl.param["limit"] = 10
ampl.param["demand"] = 6

In [15]:
# run.py
ampl.con["ServeDemand"].drop()
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.var["Flow"].value(), 6))

solved 0.0


### Another way in amplpy

Steps 1 to 4 of the checks under "Check a Nonlinear Result in Order", with the same `model.mod`
and `data.py`: `solve_result`, the `solve_message` value, `value()`, `body()` and the declared
bounds `lb0()` and `ub0()`. The recomputed objective is $6/(1-0.6)=15$, the demand row is exactly
met, and neither declared bound, 0 or 9.9, is active:

In [16]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [17]:
# data.py
ampl.param["limit"] = 10
ampl.param["demand"] = 6

In [18]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print("1.", ampl.solve_result, "|", ampl.get_value("solve_message").strip())
flow = ampl.var["Flow"].value()
print("2.", round(flow / (1 - flow / 10), 6), round(ampl.obj["CongestionCost"].value(), 6))
print("3.", round(ampl.con["ServeDemand"].body() - 6, 9))
v = ampl.var["Flow"]
print("4.", v.lb0(), round(flow, 6), v.ub0())

1. solved | Ipopt 3.12.13: Optimal Solution Found
2. 15.0 15.0
3. 0.0
4. 0.0 6.0 9.9
